# Experimentos - Ejercicio 1

Este notebook permite validar el flujo del experimento del resumidor sin depender de la API key, usando modo simulacion.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from runner import SummarizerExperiment, OUTPUT_DIR

In [ ]:
experiment = SummarizerExperiment(simulate=True)
payload = experiment.run_for_first_row()
payload['winner']

In [ ]:
results_path = OUTPUT_DIR / 'summary_experiment_results.csv'
df = pd.read_csv(results_path)
df[['model', 'quality_score', 'coverage_score', 'factuality_score', 'clarity_score', 'concision_score', 'latency_seconds']]

In [ ]:
report_path = OUTPUT_DIR / 'summary_experiment_report.md'
print(report_path.read_text(encoding='utf-8')[:1200])

## Pruebas de robustez y escalabilidad

Estas pruebas respaldan dos decisiones:
1. Ejecutar 1 fila para la consigna de la prueba tecnica.
2. Verificar que el pipeline soporta multiples filas (y por diseño puede ejecutar las 1000 filas con `--all-rows`).

In [ ]:
import time

start = time.perf_counter()
experiment_multi = SummarizerExperiment(simulate=True)
payload_multi = experiment_multi.run(rows=3)
elapsed = time.perf_counter() - start

print('executed_rows_count =', payload_multi['executed_rows_count'])
print('document_indices_executed =', payload_multi['document_indices_executed'])
print('execution_mode = simulated')
print('elapsed_seconds_for_3_rows =', round(elapsed, 4))

# Extrapolacion lineal simple para dimensionar corrida completa.
estimated_seconds_1000 = (elapsed / 3) * 1000
print('estimated_seconds_for_1000_rows_simulated =', round(estimated_seconds_1000, 2))

In [ ]:
# Verificacion estructural para demostrar que el runner soporta corrida masiva.
assert hasattr(experiment_multi, 'run')
assert payload_multi['executed_rows_count'] == 3
assert len(payload_multi['documents']) == 3
assert all(len(doc['results']) == 3 for doc in payload_multi['documents'])

print('Checks de escalabilidad basica: OK')

## Prueba real de API key OpenAI (usando columna puntos_criticos del dataset)

Esta prueba usa el mismo proyecto y el mismo `runner.py`, sin crear entorno nuevo.

Que verifica:
- conexion real con OpenAI para generar un resumen,
- uso de la columna `puntos_criticos` del dataset tecnico para evaluar el resumen,
- guardado de la evidencia de prueba en `outputs/api_key_smoke_test.json`.

Notas:
- No imprime la API key.
- Si `OPENAI_API_KEY` no esta configurada, la celda se saltea con mensaje claro.

In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

from runner import SummarizerExperiment, DATASET_PATH, OUTPUT_DIR

load_dotenv()
api_key_present = bool(os.getenv("OPENAI_API_KEY"))

if not api_key_present:
    print("SKIP: OPENAI_API_KEY no configurada. Definila en .env para correr esta prueba real.")
else:
    # Misma base de codigo del proyecto, sin crear entorno aparte.
    experiment_real = SummarizerExperiment(simulate=False)

    # Toma una fila real del dataset tecnico y usa su columna puntos_criticos.
    df_demo = pd.read_csv(DATASET_PATH)
    demo_row = df_demo.iloc[0]
    texto_demo = demo_row["texto_original"]
    puntos_criticos_demo = demo_row["puntos_criticos"]

    model_demo = "gpt-4.1-mini"
    summary_result = experiment_real.generate_summary(model=model_demo, text=texto_demo)
    judge_result = experiment_real.judge_summary(
        text=texto_demo,
        critical_points=puntos_criticos_demo,
        summary=summary_result["summary"],
    )

    metrics = {
        "quality_score": judge_result.get("quality_score"),
        "coverage_score": judge_result.get("coverage_score"),
        "factuality_score": judge_result.get("factuality_score"),
        "clarity_score": judge_result.get("clarity_score"),
        "concision_score": judge_result.get("concision_score"),
        "strengths": judge_result.get("strengths", []),
        "weaknesses": judge_result.get("weaknesses", []),
        "missing_critical_points": judge_result.get("missing_critical_points", []),
    }

    smoke_payload = {
        "dataset_row_index": 0,
        "model": model_demo,
        "summary": summary_result["summary"],
        "metrics": metrics,
    }

    OUTPUT_DIR.mkdir(exist_ok=True)
    smoke_path = OUTPUT_DIR / "api_key_smoke_test.json"
    smoke_path.write_text(json.dumps(smoke_payload, ensure_ascii=False, indent=2), encoding="utf-8")

    print("OK: llamada real ejecutada usando puntos_criticos del dataset")
    print("archivo_evidencia:", smoke_path)
    print("metricas_juez:")
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
